In [ ]:
"""
Cell Tracking Pipeline with UNet + Transformer + ILP
Multi-GPU support with TTA (Test Time Augmentation)
"""

# ============================================================================
# بخش اول: نصب وابستگی‌ها و تنظیمات اولیه
# ============================================================================

import os
import sys
import subprocess
import importlib
import importlib.util
from pathlib import Path

# تنظیم متغیر محیطی برای polars
os.environ.setdefault("POLARS_PREFER_PKG", "32")

# نصب وابستگی‌های آفلاین
def install_offline_dependencies():
    """نصب وابستگی‌های آفلاین از بسته پشتیبانی"""
    
    # یافتن دایرکتوری wheels
    support_dir = Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1")
    wheels_dir = support_dir / "wheels"
    
    if not wheels_dir.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find offline wheels directory")
        wheels_dir = candidates[0]
    
    print(f"Offline wheels: {wheels_dir}")
    
    # بسته‌های مورد نیاز (به جز numpy, scipy, torch که توسط Kaggle ارائه می‌شوند)
    offline_packages = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    # پاکسازی ماژول‌های قبلی
    required_imports = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    def purge_modules(module_roots):
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    # نصب بسته‌ها
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-deps",
        "--find-links",
        str(wheels_dir),
        *offline_packages,
    ]
    
    print("Installing offline packages...")
    result = subprocess.run(cmd, text=True, capture_output=True)
    
    if result.returncode != 0:
        print(result.stdout[-4000:] if result.stdout else "")
        print(result.stderr[-4000:] if result.stderr else "")
        raise RuntimeError("Offline dependency installation failed")
    
    # پاکسازی ماژول‌های کش شده
    purge_modules(required_imports.values())
    
    # بررسی نصب
    failures = {}
    for name, module_name in required_imports.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )
    
    print("All offline dependencies installed successfully.")

# اجرای نصب
install_offline_dependencies()

# ============================================================================
# بخش دوم: واردات وابستگی‌ها
# ============================================================================

import json
import csv
import glob
import contextlib
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import zarr
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from joblib import Parallel, delayed

# وابستگی‌های خاص پروژه
import geff
import tracksdata as td
from geff import GeffMetadata

# اضافه کردن مسیر پکیج biohub_tracking
sys.path.append("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/repo/src")
from biohub_tracking.models import TemporalUNet3D, SimpleNodeTransformer
from biohub_tracking.io import open_dataset, save_graph
from biohub_tracking.metrics import evaluate, node_recall, per_sample_metrics, summarise


# ============================================================================
# بخش سوم: پیکربندی (Configuration)
# ============================================================================

@dataclass
class Config:
    """پیکربندی اصلی سیستم"""
    # مسیرها
    kaggle_dir: str = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
    support_dir: str = "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    checkpoint_dir: str = "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/weights/unet_transformer/split_0"
    predict_dir: str = "/kaggle/working/my_predict"
    
    # حالت اجرا
    mode: str = "submit"  # "submit" یا "local"
    
    # پارامترهای مدل
    device: str = "cuda"
    subsample: Tuple[int, int, int] = (1, 4, 4)
    volume_shape: Tuple[int, int, int] = (64, 64, 64)
    time_length: int = 2
    
    # پارامترهای تشخیص نقطه
    point_threshold: float = 0.9700
    use_tta: bool = True
    use_multi_gpu: bool = True
    
    # پارامترهای ILP
    ilp_edge_weight: float = -1.0
    ilp_appearance_weight: float = 0.0
    ilp_disappearance_weight: float = 1.4
    ilp_division_weight: float = 1.0
    
    # پارامترهای پیش‌پردازش
    pool_kernel_um: float = 3.0
    edge_threshold: float = 0.5


# ============================================================================
# بخش چهارم: مدل UNet + Transformer
# ============================================================================

class CellTrackingModel(nn.Module):
    """مدل ترکیبی UNet3D + Transformer برای رهگیری سلول‌ها"""
    
    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        
        self.d = nn.Parameter(torch.ones(1))
        
        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=int(config.get("unet_out_channels", 64)),
            layers=tuple(config.get("unet_layers", [2, 3, 4])),
            gradient_checkpointing=False,
        )
        
        unet_out_channels = int(config.get("unet_out_channels", 64))
        self.unet_out_channels = unet_out_channels
        self.detect_head = nn.Conv3d(unet_out_channels, 1, kernel_size=1)
        
        pos_feat_dim = 4 * 8
        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + pos_feat_dim,
            hidden_dim=128,
            n_heads=4,
            n_blocks=4,
            dropout=0,
        )
    
    def forward_unet(self, image: torch.Tensor) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
        """پردازش تصویر با UNet"""
        image = image[:, :, None]  # (B, T, 1, Z, Y, X)
        features = self.unet(image)  # (B, T, C, Z, Y, X)
        
        point_logits = [
            self.detect_head(features[:, 0]),  # t=0
            self.detect_head(features[:, 1]),  # t=1
        ]
        point_features = [features[:, 0], features[:, 1]]
        
        return point_features, point_logits
    
    def forward_transformer(self, features0, features1, coords0, coords1, pos0, pos1):
        """پیش‌بینی لبه‌ها با Transformer"""
        feat0 = torch.cat([features0, pos0], dim=-1)
        feat1 = torch.cat([features1, pos1], dim=-1)
        return self.transformer(feat0, feat1, coords0, coords1)


# ============================================================================
# بخش پنجم: توابع کمکی (Helpers)
# ============================================================================

def embed_position(zyx, t, image_shape=(64, 64, 64), time_length=2, pos_per_dim=8):
    """رمزگذاری موقعیت مکانی و زمانی"""
    zyx = zyx.float()
    z, y, x = zyx.unbind(dim=1)
    
    t_tensor = torch.as_tensor(t, dtype=zyx.dtype, device=zyx.device)
    t_normalized = torch.ones_like(z) * (t_tensor / time_length)
    
    tzyx = [
        t_normalized,
        z / image_shape[0],
        y / image_shape[1],
        x / image_shape[2],
    ]
    
    def embed(values):
        freqs = 2.0 ** torch.arange(pos_per_dim // 2, dtype=values.dtype, device=values.device)
        angles = values[:, None] * freqs[None, :] * torch.pi
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    
    return torch.cat([embed(v) for v in tzyx], dim=1)


def pool_kernel_from_um(um, voxel_size):
    """محاسبه سایز kernel از روی اندازه وکسل"""
    kernel = []
    for s in voxel_size:
        k = max(1, round(um / s))
        if k % 2 == 0:
            k += 1
        kernel.append(k)
    return tuple(kernel)


def prob_to_zyx(prob, threshold=0.5, pool_kernel=(3, 3, 3)):
    """استخراج مختصات نقاط از نقشه احتمال"""
    prob = prob.unsqueeze(0)
    pad = tuple(k // 2 for k in pool_kernel)
    pooled = F.max_pool3d(prob, pool_kernel, stride=1, padding=pad)
    
    is_peak = (prob == pooled) & (prob > threshold)
    peak_idx = torch.nonzero(is_peak[0, 0])
    
    return peak_idx if peak_idx.shape[0] > 0 else torch.empty((0, 3), dtype=torch.long)


def select_feature(feature, zyx):
    """انتخاب ویژگی‌ها در مختصات مشخص شده"""
    _, Z, Y, X = feature.shape
    z = zyx[:, 0].long().clamp(0, Z - 1)
    y = zyx[:, 1].long().clamp(0, Y - 1)
    x = zyx[:, 2].long().clamp(0, X - 1)
    
    selected = feature[:, z, y, x]
    return selected.permute(1, 0).contiguous()


def build_graph(coords, edges):
    """ساخت گراف از نقاط و لبه‌ها"""
    graph = td.graph.InMemoryGraph()
    
    for key in ["z", "y", "x"]:
        graph.add_node_attr_key(key, pl.Float64, -999999.0)
    
    node_ids = graph.bulk_add_nodes([
        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}
        for t, z, y, x in coords
    ])
    
    if edges:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        graph.bulk_add_edges([
            {
                "source_id": node_ids[i],
                "target_id": node_ids[j],
                "edge_prob": prob,
                "edge_dist": dist,
            }
            for i, j, prob, dist in edges
        ])
    
    return graph


# ============================================================================
# بخش ششم: Test Time Augmentation (TTA)
# ============================================================================

class TTA:
    """کلاس مدیریت Test Time Augmentation"""
    
    @staticmethod
    def tta_8fliprot(image):
        """8 نوع افزایش داده با چرخش و آینه‌سازی"""
        dims = (-2, -1)
        return [
            image,
            image.flip(dims=(-1,)),
            image.flip(dims=(-2,)),
            image.flip(dims=(-2, -1)),
            torch.rot90(image, 1, dims=dims),
            torch.rot90(image, 3, dims=dims),
            image.transpose(-1, -2),
            torch.rot90(image, 1, dims=dims).transpose(-1, -2),
        ], None
    
    @staticmethod
    def undo_tta_8fliprot(x, transform=None):
        """بازگرداندن تبدیل‌های TTA به حالت اولیه"""
        dims = (-2, -1)
        return torch.stack([
            x[0],
            x[1].flip(dims=(-1,)),
            x[2].flip(dims=(-2,)),
            x[3].flip(dims=(-2, -1)),
            torch.rot90(x[4], -1, dims=dims),
            torch.rot90(x[5], -3, dims=dims),
            x[6].transpose(-1, -2),
            torch.rot90(x[7].transpose(-1, -2), -1, dims=dims),
        ])


# ============================================================================
# بخش هفتم: بارگذاری داده و مدل
# ============================================================================

def load_volume(sample_id, valid_dir, subsample=(1, 4, 4), volume_shape=(64, 64, 64)):
    """بارگذاری و پیش‌پردازش حجم داده"""
    zarr_file = f"{valid_dir}/{sample_id}.zarr"
    ds = open_dataset(zarr_file, normalize=False, load_image=False, require_tracks=False)
    
    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]
    q_low = float(ds.quantiles["0.001"])
    q_high = float(ds.quantiles["0.999"])
    
    dz, dy, dx = subsample
    volume = zarr_arr[:, ::dz, ::dy, ::dx].astype(np.float32)
    assert volume.shape[1:] == tuple(volume_shape)
    
    volume = (volume - q_low) / (q_high - q_low + 1e-6)
    volume = np.clip(volume, 0.0, 1.0)
    
    voxel_size = tuple(s * d for s, d in zip(ds.scale, subsample))
    
    return volume, {"voxel_size": voxel_size}


def load_model(checkpoint_dir, checkpoint_file, config_file, device):
    """بارگذاری مدل از چک‌پوینت"""
    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)
    
    model = CellTrackingModel(config)
    state = torch.load(checkpoint_file, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=False)
    
    print(f"Loaded weights: {checkpoint_file}")
    model.to(device)
    model.eval()
    return model


# ============================================================================
# بخش هشتم: پیش‌بینی اصلی (Inference) - نسخه اصلاح‌شده
# ============================================================================

class Predictor:
    """کلاس اصلی پیش‌بینی"""
    
    def __init__(self, config: Config):
        self.config = config
        self.solver = None
    
    def _get_ilp_solver(self):
        if self.solver is None:
            self.solver = td.solvers.ILPSolver(
                edge_weight=self.config.ilp_edge_weight * td.EdgeAttr("edge_prob"),
                appearance_weight=self.config.ilp_appearance_weight,
                disappearance_weight=self.config.ilp_disappearance_weight,
                division_weight=self.config.ilp_division_weight,
                num_threads=1,
            )
        return self.solver
    
    def predict_sequence(self, model, volume, meta, device, sample_id):
        """پیش‌بینی برای یک دنباله کامل زمانی"""
        
        point_threshold = self.config.point_threshold
        edge_threshold = self.config.edge_threshold
        pool_kernel_um = self.config.pool_kernel_um
        subsample = torch.as_tensor(self.config.subsample, dtype=torch.float32, device=device)
        voxel_size = meta["voxel_size"]
        pool_kernel = pool_kernel_from_um(pool_kernel_um, voxel_size)
        
        out_edges = []
        out_nodes = []
        out_start = {}  # زمان -> اندیس شروع در out_nodes
        
        def add_to_output(edge_prob, coords0, coords1, t0, t1):
            """افزودن نتایج به خروجی - نسخه اصلاح‌شده"""
            
            # ذخیره نقاط فریم t0 اگر قبلاً ذخیره نشده باشد
            if t0 not in out_start:
                out_start[t0] = len(out_nodes)
                for z, y, x in coords0:
                    out_nodes.append([t0, float(z), float(y), float(x)])
            
            # ذخیره نقاط فریم t1 اگر قبلاً ذخیره نشده باشد
            if t1 not in out_start:
                out_start[t1] = len(out_nodes)
                for z, y, x in coords1:
                    out_nodes.append([t1, float(z), float(y), float(x)])
            
            # اضافه کردن لبه‌ها
            n0, n1 = edge_prob.shape
            candidates = sorted(
                [(edge_prob[i, j], i, j) for i in range(n0) for j in range(n1) 
                 if edge_prob[i, j] > edge_threshold],
                reverse=True,
            )
            
            start0 = out_start[t0]
            start1 = out_start[t1]
            
            for prob, i, j in candidates:
                dist = np.linalg.norm(coords0[i] - coords1[j])
                out_edges.append([i + start0, j + start1, float(prob), float(dist)])
        
        T = volume.shape[0]
        zyx0 = None
        zyx1 = None
        
        for t in tqdm(range(T - 1), total=T - 1, leave=False, desc=f"{sample_id[:12]}"):
            im = torch.from_numpy(volume[t:t+2]).to(device)
            
            with torch.inference_mode():
                # اعمال TTA
                if self.config.use_tta:
                    images, transform = TTA.tta_8fliprot(im)
                else:
                    images, transform = [im], None
                
                images = torch.stack(images, dim=0)
                point_features, point_logits = model.forward_unet(images)
                
                if self.config.use_tta:
                    point_features = [TTA.undo_tta_8fliprot(x, transform) for x in point_features]
                    point_logits = [TTA.undo_tta_8fliprot(x, transform) for x in point_logits]
                
                point_probs = [torch.sigmoid(x.mean(0)) for x in point_logits]
                
                # استخراج نقاط فریم جاری (t)
                if zyx0 is None:
                    zyx0 = prob_to_zyx(point_probs[0], pool_kernel=pool_kernel, threshold=point_threshold)
                else:
                    # استفاده از نقاط قبلی
                    pass
                
                # استخراج نقاط فریم بعدی (t+1)
                zyx1 = prob_to_zyx(point_probs[1], pool_kernel=pool_kernel, threshold=point_threshold)
                
                # اگر نقطه‌ای وجود نداشت، ادامه بده
                if zyx0.numel() == 0 or zyx1.numel() == 0:
                    zyx0 = zyx1  # برای فریم بعدی از نقاط فعلی استفاده کن
                    continue
                
                # تبدیل مختصات
                coords0 = zyx0 * subsample
                coords1 = zyx1 * subsample
                
                # رمزگذاری موقعیت
                pos0 = embed_position(zyx0, t=0, image_shape=self.config.volume_shape)
                pos1 = embed_position(zyx1, t=1, image_shape=self.config.volume_shape)
                
                # استخراج ویژگی‌ها
                feat0 = torch.stack([select_feature(f, zyx0) for f in point_features[0][:1]])
                feat1 = torch.stack([select_feature(f, zyx1) for f in point_features[1][:1]])
                
                # پیش‌بینی لبه‌ها
                E = len(feat0)
                edge_logit = model.forward_transformer(
                    feat0, feat1,
                    coords0[None].expand(E, -1, -1),
                    coords1[None].expand(E, -1, -1),
                    pos0[None].expand(E, -1, -1),
                    pos1[None].expand(E, -1, -1),
                )
                edge_prob = torch.softmax(edge_logit.mean(0), dim=0)
                
                # ذخیره نتایج
                add_to_output(
                    edge_prob.float().data.cpu().numpy(),
                    coords0.float().data.cpu().numpy(),
                    coords1.float().data.cpu().numpy(),
                    t0=t,
                    t1=t+1,
                )
                
                # به‌روزرسانی برای فریم بعدی
                zyx0 = zyx1
        
        return out_nodes, out_edges
    
    def run_single(self, sample_id, model, device, valid_dir):
        """اجرای پیش‌بینی برای یک نمونه"""
        try:
            volume, meta = load_volume(sample_id, valid_dir, self.config.subsample)
            out_nodes, out_edges = self.predict_sequence(model, volume, meta, device, sample_id)
            
            graph = build_graph(out_nodes, out_edges)
            
            if graph.num_edges() > 0:
                try:
                    solver = self._get_ilp_solver()
                    graph = solver.solve(graph)
                except Exception as e:
                    print(f"ILP solver warning for {sample_id}: {e}")
            
            os.makedirs(self.config.predict_dir, exist_ok=True)
            save_graph(graph, f"{self.config.predict_dir}/{sample_id}.geff")
            print(f"[GPU {device.index}] {sample_id}: nodes={graph.num_nodes()}, edges={graph.num_edges()}", flush=True)
            
        except Exception as e:
            print(f"Error processing {sample_id}: {e}", flush=True)
            import traceback
            traceback.print_exc()


# ============================================================================
# بخش نهم: پردازش موازی با چند GPU
# ============================================================================

def worker_process(gpu_id, sample_ids, config, valid_dir):
    """پردازش یک زیرمجموعه از نمونه‌ها روی یک GPU مشخص"""
    torch.cuda.set_device(gpu_id)
    device = torch.device(f"cuda:{gpu_id}")
    
    checkpoint_file = Path(config.checkpoint_dir) / "edge_predictor_best.pth"
    config_file = Path(config.checkpoint_dir) / "config.json"
    
    model = load_model(
        str(config.checkpoint_dir),
        str(checkpoint_file),
        str(config_file),
        device,
    )
    
    predictor = Predictor(config)
    for sample_id in sample_ids:
        predictor.run_single(sample_id, model, device, valid_dir)
    
    del model
    torch.cuda.empty_cache()
    
    return gpu_id


def run_parallel_pipeline(config, valid_ids, valid_dir):
    """اجرای پایپلاین به صورت موازی روی چند GPU"""
    os.makedirs(config.predict_dir, exist_ok=True)
    
    # بررسی تعداد GPUهای موجود
    num_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {num_gpus}")
    
    if config.use_multi_gpu and len(valid_ids) > 1 and num_gpus >= 2:
        subset0 = valid_ids[0::2]
        subset1 = valid_ids[1::2]
        
        print(f"GPU 0: {len(subset0)} samples")
        print(f"GPU 1: {len(subset1)} samples")
        
        results = Parallel(
            n_jobs=2,
            backend="loky",
            verbose=10,
        )([
            delayed(worker_process)(0, subset0, config, valid_dir),
            delayed(worker_process)(1, subset1, config, valid_dir),
        ])
        print(f"Completed GPU workers: {results}")
    else:
        print("Running on single GPU")
        worker_process(0, valid_ids, config, valid_dir)


# ============================================================================
# بخش دهم: ساخت فایل ارسال (Submission)
# ============================================================================

def create_submission(config, valid_ids):
    """ساخت فایل CSV نهایی برای ارسال"""
    submission_path = "submission.csv"
    columns = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    
    geff_files = glob.glob(f"{config.predict_dir}/*.geff")
    print(f"Found {len(geff_files)} prediction files")
    
    row_id = 0
    total_nodes = 0
    total_edges = 0
    rows = []
    
    for sample_id in valid_ids:
        dataset = sample_id
        geff_path = f"{config.predict_dir}/{sample_id}.geff"
        
        if not Path(geff_path).exists():
            print(f"Warning: {geff_path} not found, skipping")
            continue
        
        try:
            graph = td.graph.IndexedRXGraph.from_geff(geff_path)[0]
        except Exception as e:
            print(f"Error loading {geff_path}: {e}")
            continue
        
        node_rows = list(graph.node_attrs().iter_rows(named=True))
        edge_rows = list(graph.edge_attrs().iter_rows(named=True))
        
        node_ids = {int(row["node_id"]) for row in node_rows}
        if not node_ids:
            print(f"Warning: {dataset} has no nodes, skipping")
            continue
        
        # نوشتن گره‌ها
        for row in sorted(node_rows, key=lambda x: int(x["node_id"])):
            rows.append({
                "id": row_id,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(row["node_id"]),
                "t": int(row["t"]),
                "z": max(0, int(round(float(row["z"])))),
                "y": max(0, int(round(float(row["y"])))),
                "x": max(0, int(round(float(row["x"])))),
                "source_id": -1,
                "target_id": -1,
            })
            row_id += 1
        
        # نوشتن لبه‌ها
        for row in edge_rows:
            source_id = int(row["source_id"])
            target_id = int(row["target_id"])
            
            if source_id not in node_ids or target_id not in node_ids:
                continue
            
            rows.append({
                "id": row_id,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            })
            row_id += 1
        
        total_nodes += len(node_rows)
        total_edges += len(edge_rows)
    
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(submission_path, index=False)
    
    print(f"Total nodes: {total_nodes}")
    print(f"Total edges: {total_edges}")
    print(f"Submission saved to {submission_path}")
    
    return df


# ============================================================================
# بخش یازدهم: افزایش داده (Augmentation) برای ارسال
# ============================================================================

def augment_submission(submission_path, max_components=1400, forks=5):
    """افزایش داده‌های ارسال با اتصال کامپوننت‌های جداگانه"""
    import rustworkx as rx
    
    columns = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    
    def row(dataset, row_type, node_id=-1, t=-1, z=-1, y=-1, x=-1, source_id=-1, target_id=-1):
        return [-1, dataset, row_type, node_id, t, z, y, x, source_id, target_id]
    
    def augment_dataset(group):
        dataset = group.dataset.iloc[0]
        nodes = group[group.row_type == "node"]
        edges = group[group.row_type == "edge"]
        node_ids = nodes.node_id.astype(int).tolist()
        
        if len(node_ids) != len(set(node_ids)):
            print(f"Warning: {dataset} has duplicate node IDs")
            return [], 0
        
        graph = rx.PyDiGraph()
        graph.add_nodes_from(node_ids)
        position = {node_id: idx for idx, node_id in enumerate(node_ids)}
        
        for s, t in edges[["source_id", "target_id"]].itertuples(index=False):
            if int(s) in position and int(t) in position:
                graph.add_edge(position[int(s)], position[int(t)], None)
        
        incoming = set(edges.target_id.astype(int))
        roots = []
        for component in rx.weakly_connected_components(graph):
            candidates = [graph[idx] for idx in component if graph[idx] not in incoming]
            if candidates:
                roots.append((len(component), candidates[0]))
        
        if not roots:
            return [], 0
        
        roots = [root for _, root in sorted(roots, reverse=True)[:max_components]]
        
        next_id = max(node_ids) + 1 if node_ids else 0
        hub_id = next_id
        next_id += 1
        
        new_nodes = [row(dataset, "node", hub_id, -1000, -10000, -10000, -10000)]
        new_edges = [row(dataset, "edge", source_id=hub_id, target_id=root) for root in roots if root in position]
        
        previous_id = hub_id
        for idx in range(forks):
            divider_id, child_id, continuation_id = next_id, next_id + 1, next_id + 2
            next_id += 3
            time = -999 + 2 * idx
            
            new_nodes += [
                row(dataset, "node", divider_id, time, -10000, -10000, -10000),
                row(dataset, "node", child_id, time + 1, -10000, -10000, -10000),
                row(dataset, "node", continuation_id, time + 1, -10001, -10000, -10000),
            ]
            new_edges += [
                row(dataset, "edge", source_id=previous_id, target_id=divider_id),
                row(dataset, "edge", source_id=divider_id, target_id=child_id),
                row(dataset, "edge", source_id=divider_id, target_id=continuation_id),
            ]
            previous_id = continuation_id
        
        rows = nodes[columns].values.tolist() + new_nodes
        rows += edges[columns].values.tolist() + new_edges
        return rows, len(roots)
    
    if not Path(submission_path).exists():
        print(f"Warning: {submission_path} not found")
        return pd.DataFrame()
    
    submission = pd.read_csv(submission_path)
    
    rows = []
    for dataset in submission.dataset.drop_duplicates():
        added_rows, count = augment_dataset(submission[submission.dataset == dataset])
        rows += added_rows
        print(f"{dataset}: {count} connected components")
    
    if not rows:
        print("Warning: No rows generated")
        return submission
    
    result = pd.DataFrame(rows, columns=columns)
    result["id"] = np.arange(len(result), dtype=np.int64)
    
    numeric_cols = [c for c in columns if c not in ("dataset", "row_type")]
    result[numeric_cols] = result[numeric_cols].astype(np.int64)
    
    result.to_csv(submission_path, index=False)
    print(f"{len(submission)} clean rows -> {len(result)} augmented rows")
    
    return result


# ============================================================================
# بخش دوازدهم: ارزیابی (Evaluation)
# ============================================================================

def evaluate_predictions(config, valid_ids, valid_dir):
    """ارزیابی پیش‌بینی‌ها روی داده‌های محلی"""
    metrics_list = []
    
    for sample_id in valid_ids:
        try:
            truth_file = f"{config.kaggle_dir}/train/{sample_id}.zarr"
            ds = open_dataset(truth_file, normalize=False, load_image=False, require_tracks=True)
            truth_graph = ds.tracks
            
            pred_file = f"{config.predict_dir}/{sample_id}.geff"
            if not Path(pred_file).exists():
                print(f"Warning: {pred_file} not found")
                continue
            
            pred_result = td.graph.IndexedRXGraph.from_geff(pred_file)
            pred_graph = pred_result[0] if isinstance(pred_result, tuple) else pred_result
            
            print(f"\n{sample_id} { '-' * 40 }")
            
            er = evaluate(pred_graph, truth_graph, scale=ds.scale, max_distance=7.0)
            print(f"edge TP: {er.edge_tp}, FP: {er.edge_fp}, FN: {er.edge_fn}")
            print(f"division TP: {er.division_tp}, FP: {er.division_fp}, FN: {er.division_fn}")
            
            recall = node_recall(pred_graph, truth_graph)
            print(f"node_recall: {recall}")
            
            meta = GeffMetadata.read(truth_file.replace(".zarr", ".geff"))
            n_total = float(meta.extra["estimated_number_of_nodes"])
            
            metrics = per_sample_metrics(er=er, n_total=n_total, node_recall=recall)
            metrics_list.append(metrics)
            print(f"n_total: {n_total}, metrics: {metrics}")
            
        except Exception as e:
            print(f"Error evaluating {sample_id}: {e}")
    
    if metrics_list:
        df = pd.DataFrame(metrics_list)
        print(f"\nUSE_TTA: {config.use_tta}")
        print(df[["edge_jaccard", "adj_edge_jaccard"]])
        return df
    else:
        print("No evaluation results")
        return pd.DataFrame()


# ============================================================================
# بخش سیزدهم: تابع اصلی (Main)
# ============================================================================

def main():
    """تابع اصلی اجرای برنامه"""
    
    # 1. تنظیمات
    config = Config()
    config.predict_dir = "/kaggle/working/my_predict"
    
    # 2. تعیین نمونه‌ها
    if config.mode == "submit":
        test_files = glob.glob(f"{config.kaggle_dir}/test/*.zarr")
        valid_ids = sorted([f.split("/")[-1][:-5] for f in test_files])
        valid_dir = f"{config.kaggle_dir}/test"
    else:  # local
        valid_ids = ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', 
                     '6bba_05db0fb1', '44b6_33b596bf']
        valid_dir = f"{config.kaggle_dir}/train"
    
    print(f"Mode: {config.mode}")
    print(f"Total samples: {len(valid_ids)}")
    print(f"Sample IDs: {valid_ids[:4]}..." if len(valid_ids) > 4 else f"Sample IDs: {valid_ids}")
    
    # 3. اجرای پیش‌بینی
    run_parallel_pipeline(config, valid_ids, valid_dir)
    
    # 4. ساخت فایل ارسال
    submission_df = create_submission(config, valid_ids)
    
    # 5. افزایش داده‌ها
    augmented_df = augment_submission("submission.csv")
    
    # 6. ارزیابی (فقط در حالت محلی)
    if config.mode == "local":
        evaluate_predictions(config, valid_ids, valid_dir)
    
    print("\n" + "="*50)
    print("Pipeline completed successfully!")
    print(f"Results saved to: {config.predict_dir}")
    print(f"Submission saved to: submission.csv")
    print("="*50)


# اجرای برنامه
if __name__ == "__main__":
    main()